In [ ]:
AgroVision SaaS: análisis comercial y de clientes

Caso práctico con datos sintéticos de una empresa AgTech B2B SaaS (enero de 2025 a junio de 2026, en USD). 
Objetivo: analizar la adquisición, el funnel comercial y la retención, y traducir los resultados en recomendaciones.
Herramientas: Python (pandas), SQLite y Power BI.
Contenido: 1) Calidad de datos, 2) Marketing Acquisition, 3) Sales Funnel, 4) SaaS Economics, 5) Exportación para Power BI.

In [ ]:
import os

DATA_PATH = "/kaggle/input/datasets/sebastianeyalir/agrovision-saas"

print("Archivos disponibles:")
for file in os.listdir(DATA_PATH):
    print(file)

In [3]:
import pandas as pd
import sqlite3
import os

# Crear base de datos SQLite en memoria
conn = sqlite3.connect(":memory:")

# Detectar y cargar todos los CSV
csv_files = sorted([
    f for f in os.listdir(DATA_PATH)
    if f.lower().endswith(".csv")
])

tables = {}

for file in csv_files:
    table_name = os.path.splitext(file)[0]
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.to_sql(table_name, conn, if_exists="replace", index=False)

    tables[table_name] = df

    print(f"{table_name}: {len(df)} registros | {len(df.columns)} columnas")

print(f"\nTotal de tablas cargadas: {len(tables)}")

accounts: 100 registros | 8 columnas
leads: 1800 registros | 9 columnas
marketing_campaigns: 108 registros | 7 columnas
opportunities: 535 registros | 15 columnas
product_usage_monthly: 832 registros | 9 columnas
sales_reps: 15 registros | 6 columnas
subscriptions_monthly: 832 registros | 8 columnas

Total de tablas cargadas: 7


In [4]:
# Verificar estructura de las tablas
for table in tables:
    print(f"\n{'='*50}")
    print(f"TABLA: {table}")
    print(tables[table].columns.tolist())
    print(tables[table].head(3))


TABLA: accounts
['account_id', 'account_name', 'customer_segment', 'region', 'hectares', 'created_at', 'acquisition_channel', 'primary_contact_count']
  account_id        account_name         customer_segment        region  \
0     A00001  Agro Client A00001        Agronomic Advisor       Córdoba   
1     A00002  Agro Client A00002  Mid-Market Agribusiness  Buenos Aires   
2     A00003  Agro Client A00003        Agronomic Advisor       Córdoba   

   hectares  created_at acquisition_channel  primary_contact_count  
0      5000  2026-03-31              Events                      4  
1      1200  2026-01-07             Organic                      4  
2      2500  2025-11-10         Paid Search                      3  

TABLA: leads
['lead_id', 'created_at', 'campaign_id', 'channel', 'customer_segment', 'region', 'estimated_hectares', 'bdr_id', 'lead_source_type']
  lead_id  created_at campaign_id      channel         customer_segment  \
0  L00001  2026-06-20     CMP0104  Paid Social  

In [5]:
# Validación de claves y relaciones

queries = {
    "Leads duplicados": """
        SELECT COUNT(*) - COUNT(DISTINCT lead_id)
        FROM leads
    """,
    "Campaign IDs duplicados": """
        SELECT COUNT(*) - COUNT(DISTINCT campaign_id)
        FROM marketing_campaigns
    """,
    "Accounts duplicadas": """
        SELECT COUNT(*) - COUNT(DISTINCT account_id)
        FROM accounts
    """,
    "Leads sin campaña asociada": """
        SELECT COUNT(*)
        FROM leads l
        LEFT JOIN marketing_campaigns m
        ON l.campaign_id = m.campaign_id
        WHERE m.campaign_id IS NULL
    """,
    "Oportunidades sin lead asociado": """
        SELECT COUNT(*)
        FROM opportunities o
        LEFT JOIN leads l
        ON o.lead_id = l.lead_id
        WHERE l.lead_id IS NULL
    """,
    "Oportunidades ganadas sin cuenta": """
        SELECT COUNT(*)
        FROM opportunities
        WHERE outcome = 'Won'
        AND account_id IS NULL
    """
}

for name, query in queries.items():
    result = pd.read_sql_query(query, conn)
    print(f"{name}: {result.iloc[0,0]}")

Leads duplicados: 0
Campaign IDs duplicados: 0
Accounts duplicadas: 0
Leads sin campaña asociada: 0
Oportunidades sin lead asociado: 0
Oportunidades ganadas sin cuenta: 0


In [6]:
query = """
SELECT
    'subscriptions_monthly' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT account_id || '-' || month) AS unique_keys
FROM subscriptions_monthly

UNION ALL

SELECT
    'product_usage_monthly',
    COUNT(*),
    COUNT(DISTINCT account_id || '-' || month)
FROM product_usage_monthly;
"""

pd.read_sql_query(query, conn)

,table_name,total_rows,unique_keys
0,subscriptions_monthly,832,832
1,product_usage_monthly,832,832


In [7]:
# Etapa 2: integridad referencial entre tablas
# Cada chequeo cuenta registros que rompen una relación esperada (esperado: 0)

checks = {
    # --- Cuentas <-> tablas mensuales ---
    "Subscriptions sin cuenta": """
        SELECT COUNT(*)
        FROM subscriptions_monthly s
        LEFT JOIN accounts a
            ON s.account_id = a.account_id
        WHERE a.account_id IS NULL
    """,
    "Usage sin cuenta": """
        SELECT COUNT(*)
        FROM product_usage_monthly u
        LEFT JOIN accounts a
            ON u.account_id = a.account_id
        WHERE a.account_id IS NULL
    """,
    "Claves en subscriptions sin usage": """
        SELECT COUNT(*)
        FROM subscriptions_monthly s
        LEFT JOIN product_usage_monthly u
            ON s.account_id = u.account_id
           AND s.month = u.month
        WHERE u.account_id IS NULL
    """,
    "Claves en usage sin subscriptions": """
        SELECT COUNT(*)
        FROM product_usage_monthly u
        LEFT JOIN subscriptions_monthly s
            ON s.account_id = u.account_id
           AND s.month = u.month
        WHERE s.account_id IS NULL
    """,

    # --- Cuentas <-> oportunidades ---
    "Opportunities con account_id inexistente": """
        SELECT COUNT(*)
        FROM opportunities o
        LEFT JOIN accounts a
            ON o.account_id = a.account_id
        WHERE o.account_id IS NOT NULL
          AND a.account_id IS NULL
    """,
    "Cuentas sin suscripción": """
        SELECT COUNT(*)
        FROM accounts a
        LEFT JOIN subscriptions_monthly s
            ON a.account_id = s.account_id
        WHERE s.account_id IS NULL
    """,
    "Cuentas sin oportunidad ganada": """
        SELECT COUNT(*)
        FROM accounts
        WHERE account_id NOT IN (
            SELECT account_id
            FROM opportunities
            WHERE outcome = 'Won'
              AND account_id IS NOT NULL
        )
    """,

    # --- Equipo comercial (sales_reps) ---
    "Leads con bdr_id inexistente": """
        SELECT COUNT(*)
        FROM leads l
        LEFT JOIN sales_reps r
            ON l.bdr_id = r.rep_id
        WHERE l.bdr_id IS NOT NULL
          AND r.rep_id IS NULL
    """,
    "Opportunities con bdr_id inexistente": """
        SELECT COUNT(*)
        FROM opportunities o
        LEFT JOIN sales_reps r
            ON o.bdr_id = r.rep_id
        WHERE o.bdr_id IS NOT NULL
          AND r.rep_id IS NULL
    """,
    "Opportunities con ae_id inexistente": """
        SELECT COUNT(*)
        FROM opportunities o
        LEFT JOIN sales_reps r
            ON o.ae_id = r.rep_id
        WHERE o.ae_id IS NOT NULL
          AND r.rep_id IS NULL
    """,
    "bdr_id sin rol BDR (leads)": """
        SELECT COUNT(*)
        FROM leads l
        JOIN sales_reps r
            ON l.bdr_id = r.rep_id
        WHERE r.role <> 'BDR'
    """,
    "ae_id sin rol Account Executive (opps)": """
        SELECT COUNT(*)
        FROM opportunities o
        JOIN sales_reps r
            ON o.ae_id = r.rep_id
        WHERE r.role <> 'Account Executive'
    """,
}

for name, query in checks.items():
    result = pd.read_sql_query(query, conn)
    print(f"{name}: {result.iloc[0, 0]}")

Subscriptions sin cuenta: 0
Usage sin cuenta: 0
Claves en subscriptions sin usage: 0
Claves en usage sin subscriptions: 0
Opportunities con account_id inexistente: 72
Cuentas sin suscripción: 6
Cuentas sin oportunidad ganada: 0
Leads con bdr_id inexistente: 0
Opportunities con bdr_id inexistente: 0
Opportunities con ae_id inexistente: 0
bdr_id sin rol BDR (leads): 0
ae_id sin rol Account Executive (opps): 0


In [8]:
# Detalle de las oportunidades con account_id que no está en accounts
pd.read_sql_query("""
    SELECT o.outcome,
           COUNT(*)            AS total,
           MIN(o.account_id)   AS id_min,
           MAX(o.account_id)   AS id_max
    FROM opportunities o
    LEFT JOIN accounts a
        ON o.account_id = a.account_id
    WHERE o.account_id IS NOT NULL
      AND a.account_id IS NULL
    GROUP BY o.outcome
""", conn)

,outcome,total,id_min,id_max
0,Lost,47,P00036,P01759
1,Open,25,P00108,P01776


Conclusión: oportunidades con account_id inexistente

Los chequeos de integridad detectaron 72 oportunidades con un account_id que no existe en accounts. La consulta anterior muestra que todas son perdidas (47) o abiertas (25), con IDs de formato P…, y ninguna está ganada. No hay ninguna oportunidad ganada sin cuenta.

Tratamiento en el análisis: account_id es un campo opcional que solo se usa para vincular las oportunidades ganadas con accounts y con las tablas mensuales. El funnel comercial se calcula por opportunity_id y outcome, por lo que esta particularidad no afecta ninguna métrica.

In [9]:
# Ventana temporal de cada tabla (primer y último registro)
pd.read_sql_query("""
    SELECT 'marketing_campaigns' AS tabla, 'month' AS campo,
           MIN(month) AS desde, MAX(month) AS hasta
    FROM marketing_campaigns
    UNION ALL
    SELECT 'leads', 'created_at', MIN(created_at), MAX(created_at)
    FROM leads
    UNION ALL
    SELECT 'opportunities', 'created_at', MIN(created_at), MAX(created_at)
    FROM opportunities
    UNION ALL
    SELECT 'opportunities', 'closed_at', MIN(closed_at), MAX(closed_at)
    FROM opportunities
    UNION ALL
    SELECT 'accounts', 'created_at', MIN(created_at), MAX(created_at)
    FROM accounts
    UNION ALL
    SELECT 'subscriptions_monthly', 'month', MIN(month), MAX(month)
    FROM subscriptions_monthly
    UNION ALL
    SELECT 'product_usage_monthly', 'month', MIN(month), MAX(month)
    FROM product_usage_monthly
""", conn)

,tabla,campo,desde,hasta
0,marketing_campaigns,month,2025-01,2026-06
1,leads,created_at,2025-01-01,2026-06-27
2,opportunities,created_at,2025-01-01,2026-06-24
3,opportunities,closed_at,2025-01-15,2026-08-04
4,accounts,created_at,2025-02-01,2026-08-04
5,subscriptions_monthly,month,2025-02,2026-06
6,product_usage_monthly,month,2025-02,2026-06


In [10]:
# Cuentas sin suscripción: fecha de creación de la cuenta y de cierre de su oportunidad ganada
pd.read_sql_query("""
    SELECT a.account_id,
           a.created_at AS cuenta_creada,
           o.closed_at  AS oportunidad_ganada
    FROM accounts a
    JOIN opportunities o
        ON o.account_id = a.account_id
       AND o.outcome = 'Won'
    WHERE a.account_id NOT IN (SELECT account_id FROM subscriptions_monthly)
    ORDER BY a.account_id
""", conn)

,account_id,cuenta_creada,oportunidad_ganada
0,A00020,2026-07-18,2026-07-18
1,A00040,2026-07-27,2026-07-27
2,A00045,2026-07-01,2026-07-01
3,A00055,2026-07-05,2026-07-05
4,A00080,2026-07-22,2026-07-22
5,A00096,2026-08-04,2026-08-04


Conclusión: ventana temporal y cuentas sin suscripción

Marketing, leads y creación de oportunidades cubren de enero de 2025 a junio de 2026. Las suscripciones y el uso mensual cubren de febrero de 2025 (mes de creación de la primera cuenta) a junio de 2026.
Los cierres de oportunidades y la creación de cuentas llegan hasta el 4 de agosto de 2026, más allá del último mes de suscripciones.
6 cuentas no tienen suscripciones porque se crearon, junto con su oportunidad ganada, entre el 1 de julio y el 4 de agosto de 2026.

Tratamiento en el análisis: el período de análisis es enero de 2025 a junio de 2026. Las 6 cuentas posteriores a junio se excluyen de las métricas de MRR, churn y retención, que se calculan solo con los datos de suscripción disponibles.

In [11]:
# Marketing Acquisition: inversión, alcance y eficiencia de medios por canal
pd.read_sql_query("""
    SELECT channel,
           COUNT(*)                                          AS campañas,
           ROUND(SUM(spend_usd), 2)                          AS inversion_usd,
           SUM(impressions)                                  AS impresiones,
           SUM(clicks)                                       AS clics,
           ROUND(100.0 * SUM(clicks) / SUM(impressions), 2)  AS ctr_pct,
           ROUND(SUM(spend_usd) / SUM(clicks), 2)            AS cpc_usd
    FROM marketing_campaigns
    GROUP BY channel
    ORDER BY inversion_usd DESC
""", conn)

,channel,campañas,inversion_usd,impresiones,clics,ctr_pct,cpc_usd
0,Events,18,90766.17,442597,10702,2.42,8.48
1,Paid Search,18,75085.58,445016,19845,4.46,3.78
2,Paid Social,18,54397.38,1169677,20445,1.75,2.66
3,Webinar,18,42168.92,314472,9749,3.10,4.33
4,Organic,18,26779.33,597508,22490,3.76,1.19
5,Referral,18,16694.00,195789,11329,5.79,1.47


In [12]:
# Marketing Acquisition: leads generados y costo por lead (CPL) por canal (con total)
pd.read_sql_query("""
    WITH inversion AS (
        SELECT channel, SUM(spend_usd) AS inversion_usd
        FROM marketing_campaigns
        GROUP BY channel
    ),
    leads_canal AS (
        SELECT channel, COUNT(*) AS leads
        FROM leads
        GROUP BY channel
    ),
    por_canal AS (
        SELECT i.channel, i.inversion_usd, lc.leads
        FROM inversion i
        JOIN leads_canal lc
            ON i.channel = lc.channel
    )
    SELECT channel,
           ROUND(inversion_usd, 2)           AS inversion_usd,
           leads,
           ROUND(inversion_usd / leads, 2)   AS cpl_usd
    FROM (
        SELECT channel, inversion_usd, leads, 0 AS orden
        FROM por_canal
        UNION ALL
        SELECT 'TOTAL', SUM(inversion_usd), SUM(leads), 1
        FROM por_canal
    )
    ORDER BY orden, inversion_usd DESC
""", conn)

,channel,inversion_usd,leads,cpl_usd
0,Events,90766.17,303,299.56
1,Paid Search,75085.58,434,173.01
2,Paid Social,54397.38,326,166.86
3,Webinar,42168.92,265,159.13
4,Organic,26779.33,250,107.12
5,Referral,16694.00,222,75.20
6,TOTAL,305891.38,1800,169.94


In [13]:
# Marketing Acquisition: clientes ganados, conversión y CAC de marketing por canal (con total)
df_marketing = pd.read_sql_query("""
    WITH inversion AS (
        SELECT channel, SUM(spend_usd) AS inversion_usd
        FROM marketing_campaigns
        GROUP BY channel
    ),
    resultado AS (
        SELECT l.channel,
               COUNT(*)               AS leads,
               SUM(o.outcome = 'Won') AS clientes_ganados
        FROM leads l
        LEFT JOIN opportunities o
            ON o.lead_id = l.lead_id
        GROUP BY l.channel
    ),
    por_canal AS (
        SELECT i.channel, i.inversion_usd, r.leads, r.clientes_ganados
        FROM inversion i
        JOIN resultado r
            ON i.channel = r.channel
    )
    SELECT channel,
           ROUND(inversion_usd, 2)                        AS inversion_usd,
           leads,
           clientes_ganados,
           ROUND(100.0 * clientes_ganados / leads, 2)     AS conv_lead_a_cliente_pct,
           ROUND(inversion_usd / clientes_ganados, 2)     AS cac_marketing_usd
    FROM (
        SELECT channel, inversion_usd, leads, clientes_ganados, 0 AS orden
        FROM por_canal
        UNION ALL
        SELECT 'TOTAL', SUM(inversion_usd), SUM(leads), SUM(clientes_ganados), 1
        FROM por_canal
    )
    ORDER BY orden, cac_marketing_usd
""", conn)
df_marketing



,channel,inversion_usd,leads,clientes_ganados,conv_lead_a_cliente_pct,cac_marketing_usd
0,Referral,16694.00,222,14,6.31,1192.43
1,Organic,26779.33,250,17,6.80,1575.25
2,Webinar,42168.92,265,19,7.17,2219.42
3,Paid Search,75085.58,434,23,5.30,3264.59
4,Events,90766.17,303,18,5.94,5042.56
5,Paid Social,54397.38,326,9,2.76,6044.15
6,TOTAL,305891.38,1800,100,5.56,3058.91


Conclusión: Marketing Acquisition

La inversión total fue de 305.891 USD, que generó 1.800 leads y 100 clientes: conversión de lead a cliente del 5,56% y CAC de marketing de 3.059 USD.
Referral, Organic y Webinar combinan conversión superior al promedio y CAC por debajo de él. Paid Social convierte 2,76% y tiene un CAC de 6.044 USD, casi el doble del promedio. Events tiene el CPL y el CAC más altos aun con conversión media.
El CPL no alcanza como criterio de asignación de presupuesto: Paid Social tiene un CPL intermedio y el peor CAC.
Limitaciones: el CAC incluye solo inversión de marketing (sin sueldos comerciales); la relación entre canal y conversión es una correlación observada, no una causalidad; y las diferencias pequeñas entre canales (por ejemplo, Webinar frente a Referral) no son concluyentes, a diferencia de la brecha de Paid Social.
Hipótesis para validar: revisar la inversión en Paid Social y Events, y evaluar si Referral, Organic y Webinar pueden escalar sin perder conversión.

In [14]:
# Sales Funnel: conversión entre etapas, de lead a cliente ganado
df_funnel = pd.read_sql_query("""
    WITH etapas AS (
        SELECT 1 AS orden, 'Leads' AS etapa, COUNT(*) AS n FROM leads
        UNION ALL
        SELECT 2, 'Qualified',  COUNT(qualified_at)  FROM opportunities
        UNION ALL
        SELECT 3, 'Demo',       COUNT(demo_at)       FROM opportunities
        UNION ALL
        SELECT 4, 'Proposal',   COUNT(proposal_at)   FROM opportunities
        UNION ALL
        SELECT 5, 'Closed Won', SUM(outcome = 'Won') FROM opportunities
    )
    SELECT etapa,
           n,
           ROUND(100.0 * n / LAG(n) OVER (ORDER BY orden), 2) AS conv_vs_etapa_anterior_pct,
           ROUND(100.0 * n / FIRST_VALUE(n) OVER (ORDER BY orden), 2) AS conv_vs_leads_pct
    FROM etapas
    ORDER BY orden
""", conn)
df_funnel

,etapa,n,conv_vs_etapa_anterior_pct,conv_vs_leads_pct
0,Leads,1800,NaN,100.00
1,Qualified,535,29.72,29.72
2,Demo,467,87.29,25.94
3,Proposal,352,75.37,19.56
4,Closed Won,100,28.41,5.56


In [15]:
# Sales Funnel: estado actual de las oportunidades (ganadas, perdidas y en curso)
pd.read_sql_query("""
    WITH resultado AS (
        SELECT outcome, current_stage, COUNT(*) AS oportunidades
        FROM opportunities
        GROUP BY outcome, current_stage
    )
    SELECT outcome,
           current_stage,
           oportunidades,
           ROUND(100.0 * oportunidades / (SELECT SUM(oportunidades) FROM resultado), 2) AS pct_del_total
    FROM (
        SELECT outcome, current_stage, oportunidades, 0 AS orden
        FROM resultado
        UNION ALL
        SELECT 'TOTAL', '', SUM(oportunidades), 1
        FROM resultado
    )
    ORDER BY orden, oportunidades DESC
""", conn)

,outcome,current_stage,oportunidades,pct_del_total
0,Lost,Closed Lost,252,47.10
1,Open,Demo,115,21.50
2,Won,Closed Won,100,18.69
3,Open,Qualified,68,12.71
4,TOTAL,,535,100.00


In [16]:
# Sales Funnel: motivos de pérdida de las oportunidades perdidas
df_motivos = pd.read_sql_query("""
    WITH motivos AS (
        SELECT lost_reason, COUNT(*) AS oportunidades
        FROM opportunities
        WHERE outcome = 'Lost'
        GROUP BY lost_reason
    )
    SELECT lost_reason,
           oportunidades,
           ROUND(100.0 * oportunidades / (SELECT SUM(oportunidades) FROM motivos), 2) AS pct_de_perdidas
    FROM (
        SELECT lost_reason, oportunidades, 0 AS orden
        FROM motivos
        UNION ALL
        SELECT 'TOTAL', SUM(oportunidades), 1
        FROM motivos
    )
    ORDER BY orden, oportunidades DESC
""", conn)
df_motivos

,lost_reason,oportunidades,pct_de_perdidas
0,Price,63,25.00
1,No decision,59,23.41
2,Missing functionality,49,19.44
3,No response,49,19.44
4,Competitor,32,12.70
5,TOTAL,252,100.00


In [17]:
# Sales Funnel: ciclo de venta y valor de las oportunidades cerradas
pd.read_sql_query("""
    SELECT CASE outcome WHEN 'Won' THEN 'Ganadas' ELSE 'Perdidas' END AS resultado,
           COUNT(*) AS oportunidades,
           ROUND(AVG(julianday(closed_at) - julianday(created_at)), 1) AS ciclo_dias_prom,
           ROUND(SUM(deal_mrr_usd), 2) AS mrr_deals_usd,
           ROUND(AVG(deal_mrr_usd), 2) AS mrr_prom_por_deal_usd
    FROM opportunities
    WHERE outcome IN ('Won', 'Lost')
    GROUP BY outcome
    ORDER BY outcome DESC
""", conn)

,resultado,oportunidades,ciclo_dias_prom,mrr_deals_usd,mrr_prom_por_deal_usd
0,Ganadas,100,33.5,51278.68,512.79
1,Perdidas,252,29.3,156563.68,621.28


Conclusión: Sales Funnel

De 1.800 leads, 535 llegan a Qualified (29,7%), 467 a Demo, 352 a Proposal y 100 se ganan: conversión total del 5,56%. Una vez calificado el lead, el avance es alto (87% a Demo y 75% a Propuesta). Las dos etapas críticas son la calificación inicial y el cierre.
Win rate: 28,4% (100 ganadas sobre 352 cerradas). Hay 183 oportunidades abiertas (en Demo y Qualified) que no entran en el cálculo.
Motivos de pérdida: precio 25,0%, sin decisión 23,4%, falta de funcionalidad 19,4%, sin respuesta 19,4% y competidor 12,7%. Casi 43% de las pérdidas no tienen un rechazo explícito.
Ciclo y valor: ciclo promedio de 33,5 días en las ganadas y 29,3 en las perdidas. MRR ganado de 51.279 USD (513 USD por deal) y MRR potencial no concretado de 156.564 USD (621 USD por deal). El win rate por valor es 24,7%.
Limitaciones: es una foto del pipeline y no un seguimiento por cohortes; el motivo registrado no explica la causa; el MRR de las perdidas es potencial y no ingreso; las asociaciones observadas no prueban causalidad.
Hipótesis a validar y recomendaciones: (1) registrar los criterios de calificación y la actividad comercial para poder medirlos; (2) reforzar la cadencia de seguimiento posterior a la propuesta, donde se concentran las pérdidas sin respuesta ni decisión; (3) revisar la propuesta de valor y el pricing en los deals de mayor MRR, que pesan más entre las perdidas; (4) llevar a producto el 19% de pérdidas por falta de funcionalidad.

In [18]:
# SaaS Economics: cuentas dadas de baja y control de reactivaciones
pd.read_sql_query("""
    WITH baja AS (
        SELECT account_id, MIN(month) AS mes_baja
        FROM subscriptions_monthly
        WHERE status = 'Churned'
        GROUP BY account_id
    )
    SELECT COUNT(*) AS cuentas_dadas_de_baja,
           MIN(mes_baja) AS primera_baja,
           MAX(mes_baja) AS ultima_baja,
           SUM(EXISTS (
               SELECT 1
               FROM subscriptions_monthly s
               WHERE s.account_id = baja.account_id
                 AND s.status = 'Active'
                 AND s.month > baja.mes_baja
           ))             AS reactivadas
    FROM baja
""", conn)

,cuentas_dadas_de_baja,primera_baja,ultima_baja,reactivadas
0,20,2025-08,2026-06,0


In [19]:
# SaaS Economics: churn mensual de clientes (cuentas que se dan de baja / cuentas activas al inicio del mes)
df_churn = pd.read_sql_query("""
    WITH activas AS (
        SELECT month, COUNT(*) AS cuentas_activas
        FROM subscriptions_monthly
        WHERE status = 'Active'
        GROUP BY month
    ),
    baja AS (
        SELECT account_id, MIN(month) AS mes_baja
        FROM subscriptions_monthly
        WHERE status = 'Churned'
        GROUP BY account_id
    ),
    bajas_mes AS (
        SELECT mes_baja AS month, COUNT(*) AS bajas
        FROM baja
        GROUP BY mes_baja
    ),
    base AS (
        SELECT a.month,
               LAG(a.cuentas_activas) OVER (ORDER BY a.month) AS activas_inicio_mes,
               COALESCE(b.bajas, 0) AS bajas
        FROM activas a
        LEFT JOIN bajas_mes b
            ON a.month = b.month
    )
    SELECT month,
           activas_inicio_mes,
           bajas,
           ROUND(100.0 * bajas / activas_inicio_mes, 2) AS churn_mensual_pct
    FROM (
        SELECT month, activas_inicio_mes, bajas, 0 AS orden
        FROM base
        WHERE activas_inicio_mes IS NOT NULL
        UNION ALL
        SELECT 'TOTAL', SUM(activas_inicio_mes), SUM(bajas), 1
        FROM base
        WHERE activas_inicio_mes IS NOT NULL
    )
    ORDER BY orden, month
""", conn)
df_churn

,month,activas_inicio_mes,bajas,churn_mensual_pct
0,2025-03,7,0,0.00
1,2025-04,11,0,0.00
2,2025-05,17,0,0.00
3,2025-06,23,0,0.00
4,2025-07,27,0,0.00
5,2025-08,34,1,2.94
6,2025-09,36,1,2.78
7,2025-10,40,1,2.50
8,2025-11,41,1,2.44
9,2025-12,47,1,2.13


In [20]:
# SaaS Economics: ingreso mensual (average revenue per account) y margen bruto de las cuentas activas
pd.read_sql_query("""
    SELECT COUNT(*) AS cuentas_mes_activas,
           COUNT(DISTINCT account_id) AS cuentas_distintas,
           ROUND(SUM(mrr_usd), 2) AS mrr_acumulado_usd,
           ROUND(SUM(mrr_usd) / COUNT(*), 2) AS mrr_por_cuenta_mensual_usd,
           ROUND(100.0 * SUM(mrr_usd * gross_margin_pct) / SUM(mrr_usd), 2) AS margen_bruto_pct
    FROM subscriptions_monthly
    WHERE status = 'Active'
""", conn)

,cuentas_mes_activas,cuentas_distintas,mrr_acumulado_usd,mrr_por_cuenta_mensual_usd,margen_bruto_pct
0,747,94,414219.36,554.51,79.89


In [21]:
# SaaS Economics: LTV, LTV/CAC y payback del CAC (estimados), con dos escenarios de churn
df_unit = pd.read_sql_query("""
    WITH activas AS (
        SELECT month, COUNT(*) AS cuentas_activas
        FROM subscriptions_monthly
        WHERE status = 'Active'
        GROUP BY month
    ),
    baja AS (
        SELECT account_id, MIN(month) AS mes_baja
        FROM subscriptions_monthly
        WHERE status = 'Churned'
        GROUP BY account_id
    ),
    bajas_mes AS (
        SELECT mes_baja AS month, COUNT(*) AS bajas
        FROM baja
        GROUP BY mes_baja
    ),
    base AS (
        SELECT a.month,
               LAG(a.cuentas_activas) OVER (ORDER BY a.month) AS activas_inicio_mes,
               COALESCE(b.bajas, 0)                           AS bajas
        FROM activas a
        LEFT JOIN bajas_mes b
            ON a.month = b.month
    ),
    churn AS (
        SELECT 'Promedio del período' AS escenario,
               1.0 * SUM(bajas) / SUM(activas_inicio_mes) AS churn_mensual
        FROM base
        WHERE activas_inicio_mes IS NOT NULL
        UNION ALL
        SELECT 'Últimos 2 meses',
               1.0 * SUM(bajas) / SUM(activas_inicio_mes)
        FROM base
        WHERE month >= '2026-05'
    ),
    supuestos AS (
        SELECT (SELECT SUM(spend_usd) FROM marketing_campaigns)
             / (SELECT COUNT(*) FROM opportunities WHERE outcome = 'Won')  AS cac_marketing,
               (SELECT SUM(mrr_usd) / COUNT(*)
                FROM subscriptions_monthly WHERE status = 'Active')        AS arpa,
               (SELECT SUM(mrr_usd * gross_margin_pct) / SUM(mrr_usd)
                FROM subscriptions_monthly WHERE status = 'Active')        AS margen
    )
    SELECT c.escenario,
           ROUND(100 * c.churn_mensual, 2)                                 AS churn_mensual_pct,
           ROUND(s.arpa, 2)                                                AS ingreso_por_cuenta_usd,
           ROUND(100 * s.margen, 2)                                        AS margen_bruto_pct,
           ROUND(s.cac_marketing, 2)                                       AS cac_marketing_usd,
           ROUND(s.arpa * s.margen / c.churn_mensual, 0)                   AS ltv_usd,
           ROUND(s.arpa * s.margen / c.churn_mensual / s.cac_marketing, 2) AS ltv_cac,
           ROUND(s.cac_marketing / (s.arpa * s.margen), 1)                 AS payback_meses
    FROM churn c
    CROSS JOIN supuestos s
""", conn)
df_unit

,escenario,churn_mensual_pct,ingreso_por_cuenta_usd,margen_bruto_pct,cac_marketing_usd,ltv_usd,ltv_cac,payback_meses
0,Promedio del período,2.97,554.51,79.89,3058.91,14908.0,4.87,6.9
1,Últimos 2 meses,6.16,554.51,79.89,3058.91,7187.0,2.35,6.9


Conclusión: SaaS Economics

Churn: 20 cuentas se dieron de baja entre agosto de 2025 y junio de 2026, ninguna se reactivó. El churn mensual promedio ponderado es 2,97%, pero en mayo y junio de 2026 subió a 6,16% (9 de las 20 bajas).
Unit economics (estimados): ARPA (ingreso promedio por cuenta al mes) de 554,51 USD por cuenta por mes, margen bruto del 79,89% y CAC de marketing de 3.059 USD. Payback de 6,9 meses. LTV de 14.908 USD (LTV/CAC 4,87) con el churn promedio, y de 7.187 USD (LTV/CAC 2,35) con el churn de los últimos 2 meses.
Lectura: el CAC se recupera rápido, pero la rentabilidad por cliente depende de la retención: si el churn reciente se mantiene, el LTV/CAC cae por debajo de la referencia habitual de 3.
Limitaciones: el CAC incluye solo inversión de marketing; el LTV supone churn y ARPA constantes y extrapola más allá del período observado; las bajas recientes son pocas para confirmar una tendencia.
Recomendaciones: (1) investigar la causa de las bajas de mayo y junio de 2026 y monitorear el churn mensual como KPI; (2) priorizar la retención antes de escalar la inversión; (3) calcular el CAC con los costos comerciales para tener la rentabilidad completa.

In [22]:
import os

# Exportar los resultados a CSV para Power BI (sin las filas TOTAL)
salida = "/kaggle/working/powerbi"
os.makedirs(salida, exist_ok=True)

exportar = {
    "funnel_etapas.csv":   df_funnel,
    "marketing_canal.csv": df_marketing[df_marketing["channel"] != "TOTAL"],
    "motivos_perdida.csv": df_motivos[df_motivos["lost_reason"] != "TOTAL"],
    "churn_mensual.csv":   df_churn[df_churn["month"] != "TOTAL"],
    "unit_economics.csv":  df_unit,
}

for nombre, df in exportar.items():
    df.to_csv(os.path.join(salida, nombre), index=False, encoding="utf-8-sig")
    print(f"{nombre}: {len(df)} filas")

funnel_etapas.csv: 5 filas
marketing_canal.csv: 6 filas
motivos_perdida.csv: 5 filas
churn_mensual.csv: 16 filas
unit_economics.csv: 2 filas
